In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import warnings, gc, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

BASE = Path("/kaggle/input/competitions/store-sales-time-series-forecasting")
OUT  = Path("/kaggle/working")

# ================================================================
# 1. LOAD DATA
# ================================================================
print("=" * 60)
print("STEP 1 — Loading Data")
print("=" * 60)

train  = pd.read_csv(BASE/"train.csv",           parse_dates=["date"])
test   = pd.read_csv(BASE/"test.csv",            parse_dates=["date"])
stores = pd.read_csv(BASE/"stores.csv")
oil    = pd.read_csv(BASE/"oil.csv",             parse_dates=["date"])
hol    = pd.read_csv(BASE/"holidays_events.csv", parse_dates=["date"])
trans  = pd.read_csv(BASE/"transactions.csv",    parse_dates=["date"])

train["sales"]       = train["sales"].clip(lower=0).astype("float32")
train["log1p_sales"] = np.log1p(train["sales"]).astype("float32")
train["onpromotion"] = train["onpromotion"].astype("int16")
test["onpromotion"]  = test["onpromotion"].astype("int16")

print(f"  Train: {train.shape} | {train.date.min().date()} → {train.date.max().date()}")
print(f"  Test : {test.shape}  | {test.date.min().date()} → {test.date.max().date()}")
print(f"  Test window = {(test.date.max()-test.date.min()).days+1} days")
print(f"  ✅ Safe minimum lag = 16 days (lag_16 of Aug31 = Aug15 = last train day)")

# ================================================================
# 2. LOOKUP TABLES
# ================================================================
print("\nSTEP 2 — Building lookup tables")

# ── stores ──
stores = stores.rename(columns={"type": "store_type"})
le_type = LabelEncoder().fit(stores["store_type"])
stores["store_type_enc"] = le_type.transform(stores["store_type"]).astype("int8")
stores["cluster"]        = stores["cluster"].astype("int8")

# ── oil (interpolate, rolling) ──
oil_rng  = pd.date_range(oil.date.min(), "2017-08-31", freq="D")
oil_full = oil.set_index("date").reindex(oil_rng).rename_axis("date").reset_index()
oil_full["dcoilwtico"] = oil_full["dcoilwtico"].interpolate("linear").ffill().bfill()
oil_full["oil_7d"]     = oil_full["dcoilwtico"].rolling(7,  min_periods=1).mean()
oil_full["oil_28d"]    = oil_full["dcoilwtico"].rolling(28, min_periods=1).mean()
oil_full["oil_diff"]   = oil_full["dcoilwtico"].diff().fillna(0)
for col in ["dcoilwtico","oil_7d","oil_28d","oil_diff"]:
    oil_full[col] = oil_full[col].astype("float32")

# ── holidays ──
nat_dates   = set(hol[hol.locale=="National"]["date"])
transferred = set(hol[hol.transferred==True]["date"])
reg_map     = hol[hol.locale=="Regional"].groupby("date")["locale_name"].apply(set).to_dict()
loc_map     = hol[hol.locale=="Local"].groupby("date")["locale_name"].apply(set).to_dict()
nat_list    = sorted(nat_dates)

all_days  = pd.date_range("2013-01-01", "2017-08-31", freq="D")
hol_lookup = pd.DataFrame({"date": all_days})
hol_lookup["is_national_holiday"] = hol_lookup.date.isin(nat_dates).astype("int8")
hol_lookup["is_transferred"]      = hol_lookup.date.isin(transferred).astype("int8")
def nearest_hol(d):
    return min([(d-h).days for h in nat_list], key=abs) if nat_list else 0
hol_lookup["days_to_holiday"] = hol_lookup.date.map(nearest_hol).astype("int16")

# ── transactions ──
trans = trans.sort_values(["store_nbr","date"])
gtx   = trans.groupby("store_nbr")["transactions"]
trans["tx_7d"]  = gtx.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
trans["tx_28d"] = gtx.transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
for col in ["transactions","tx_7d","tx_28d"]:
    trans[col] = trans[col].astype("float32")

# ── family encoder ──
le_fam = LabelEncoder().fit(train["family"].astype(str))

print("  Done ✓")

# ================================================================
# 3. FEATURE ENGINEERING FUNCTION
#    Called on both train and test after lag features are added
# ================================================================
def add_all_features(df):
    d = df["date"]
    # Calendar
    df["year"]           = d.dt.year.astype("int16")
    df["month"]          = d.dt.month.astype("int8")
    df["day"]            = d.dt.day.astype("int8")
    df["dayofweek"]      = d.dt.dayofweek.astype("int8")
    df["dayofyear"]      = d.dt.dayofyear.astype("int16")
    df["weekofyear"]     = d.dt.isocalendar().week.astype("int8").values
    df["quarter"]        = d.dt.quarter.astype("int8")
    df["is_weekend"]     = (d.dt.dayofweek >= 5).astype("int8")
    df["is_month_start"] = d.dt.is_month_start.astype("int8")
    df["is_month_end"]   = d.dt.is_month_end.astype("int8")
    df["is_payday"]      = ((d.dt.day == 15) | d.dt.is_month_end).astype("int8")
    df["month_sin"]      = np.sin(2*np.pi*df.month/12).astype("float32")
    df["month_cos"]      = np.cos(2*np.pi*df.month/12).astype("float32")
    df["dow_sin"]        = np.sin(2*np.pi*df.dayofweek/7).astype("float32")
    df["dow_cos"]        = np.cos(2*np.pi*df.dayofweek/7).astype("float32")
    df["days_since"]     = (d - pd.Timestamp("2013-01-01")).dt.days.astype("int16")

    # Store meta
    df = df.merge(stores[["store_nbr","store_type_enc","cluster","city","state"]],
                  on="store_nbr", how="left")

    # Holidays
    df = df.merge(hol_lookup, on="date", how="left")
    df["is_regional_holiday"] = df.apply(
        lambda r: int(r.state in reg_map.get(r.date, set())), axis=1).astype("int8")
    df["is_local_holiday"]    = df.apply(
        lambda r: int(r.city  in loc_map.get(r.date, set())), axis=1).astype("int8")
    df["is_any_holiday"]      = (
        df.is_national_holiday | df.is_regional_holiday | df.is_local_holiday
    ).astype("int8")
    df.drop(columns=["city","state"], inplace=True)

    # Oil
    df = df.merge(oil_full[["date","dcoilwtico","oil_7d","oil_28d","oil_diff"]],
                  on="date", how="left")

    # Transactions
    df = df.merge(trans[["date","store_nbr","transactions","tx_7d","tx_28d"]],
                  on=["date","store_nbr"], how="left")

    # Family / store encoding
    df["family_enc"] = le_fam.transform(df.family.astype(str)).astype("int8")
    df["store_fam"]  = (df.store_nbr.astype("int16") * 100 +
                        df.family_enc.astype("int16")).astype("int16")

    return df

# ================================================================
# 4. LAG FEATURE COMPUTATION
#    Strategy: group by store+family, shift on log1p_sales
#    Use ONLY lags >= 16 (safe for all test dates)
#    Use same-weekday-last-year averages as proxy for short lags
# ================================================================
print("\nSTEP 3 — Computing lag features")
t0 = time.time()

#  ── Process per family to keep RAM low ──
families = train["family"].unique()
chunks   = []

for i, fam in enumerate(families):
    f = train[train.family == fam].copy()
    f = f.sort_values(["store_nbr","date"]).reset_index(drop=True)
    grp = f.groupby("store_nbr")["log1p_sales"]

    # Safe lags (>= 16 days)
    for lag in [16, 21, 28, 35, 42, 56, 91, 112, 364, 365, 366]:
        f[f"lag_{lag}"] = grp.shift(lag)

    # Same-weekday proxies (replace lag_7, lag_14)
    # lag_sw1 = same weekday 1 week ago (lag_7 proxy via 4-week avg)
    f["lag_sw_4w"]  = (grp.shift(28) + grp.shift(35) + grp.shift(42) + grp.shift(49)) / 4
    f["lag_sw_ly"]  = (grp.shift(357) + grp.shift(364) + grp.shift(371)) / 3  # last year same weekday

    # Rolling windows (shifted by 16 to avoid leakage)
    for w in [7, 14, 28, 56]:
        sh = grp.shift(16)   # shift 16 so even lag_16 is safe
        f[f"roll_mean_{w}"] = sh.transform(lambda x: x.rolling(w, min_periods=1).mean())
        f[f"roll_std_{w}"]  = sh.transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
        f[f"roll_max_{w}"]  = sh.transform(lambda x: x.rolling(w, min_periods=1).max())
        f[f"roll_min_{w}"]  = sh.transform(lambda x: x.rolling(w, min_periods=1).min())

    f["expanding_mean"] = grp.shift(16).transform(lambda x: x.expanding(1).mean())
    chunks.append(f)

    if (i+1) % 6 == 0:
        print(f"  {i+1}/{len(families)} families  {time.time()-t0:.0f}s")

train_feat = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()
print(f"  Lag done in {time.time()-t0:.0f}s | shape: {train_feat.shape}")

# ── Build TEST lag features (look back into train) ──
print("  Building test lag features …")
test_chunks = []
for fam in families:
    tr_f  = train[train.family==fam].sort_values(["store_nbr","date"])
    te_f  = test[test.family==fam].copy()
    # For test (Aug 16-31), lag_N looks back N days into train (if N>=16)
    for store in te_f.store_nbr.unique():
        tr_s  = tr_f[tr_f.store_nbr==store].set_index("date")["log1p_sales"]
        te_s  = te_f[te_f.store_nbr==store].copy()
        for lag in [16, 21, 28, 35, 42, 56, 91, 112, 364, 365, 366]:
            te_s[f"lag_{lag}"] = te_s["date"].map(
                lambda d: float(tr_s.get(d - pd.Timedelta(days=lag), np.nan)))
        for lag_name, offsets in [("lag_sw_4w",[28,35,42,49]),("lag_sw_ly",[357,364,371])]:
            vals = []
            for d in te_s["date"]:
                v = [float(tr_s.get(d-pd.Timedelta(days=o), np.nan)) for o in offsets]
                v = [x for x in v if not np.isnan(x)]
                vals.append(np.mean(v) if v else np.nan)
            te_s[lag_name] = vals
        for w in [7, 14, 28, 56]:
            for stat in ["mean","std","max","min"]:
                vals = []
                for d in te_s["date"]:
                    end   = d - pd.Timedelta(days=16)
                    start = end - pd.Timedelta(days=w-1)
                    sl    = tr_s[(tr_s.index >= start) & (tr_s.index <= end)].dropna()
                    if len(sl) == 0:
                        vals.append(np.nan)
                    elif stat == "mean": vals.append(float(sl.mean()))
                    elif stat == "std":  vals.append(float(sl.std()) if len(sl)>1 else 0.0)
                    elif stat == "max":  vals.append(float(sl.max()))
                    elif stat == "min":  vals.append(float(sl.min()))
                te_s[f"roll_{stat}_{w}"] = vals
        te_s["expanding_mean"] = te_s["date"].map(lambda d: float(
            tr_s[tr_s.index < d - pd.Timedelta(days=15)].mean()
            if len(tr_s[tr_s.index < d - pd.Timedelta(days=15)]) > 0 else np.nan))
        test_chunks.append(te_s)

test_feat = pd.concat(test_chunks, ignore_index=True)
del test_chunks; gc.collect()
print(f"  Test lag done | shape: {test_feat.shape}")

# ================================================================
# 5. ADD CALENDAR / HOLIDAY / OIL / STORE FEATURES
# ================================================================
print("\nSTEP 4 — Adding calendar, holiday, oil features")
train_feat["family"] = train_feat["family"].astype(str)
test_feat["family"]  = test_feat["family"].astype(str)
train_feat = add_all_features(train_feat)
test_feat  = add_all_features(test_feat)
gc.collect()
print(f"  Train: {train_feat.shape}  Test: {test_feat.shape}")

# ================================================================
# 6. DEFINE FEATURES
# ================================================================
DROP = {"id","date","family","sales","log1p_sales","store_type",
        "transactions","city","state"}
LAG_COLS = ([f"lag_{l}" for l in [16,21,28,35,42,56,91,112,364,365,366]] +
            ["lag_sw_4w","lag_sw_ly"] +
            [f"roll_{s}_{w}" for s in ["mean","std","max","min"] for w in [7,14,28,56]] +
            ["expanding_mean"])
FEAT_COLS = [c for c in train_feat.columns if c not in DROP]
print(f"\n  Total features: {len(FEAT_COLS)}")
print(f"  Feature list: {FEAT_COLS}")

# ================================================================
# 7. TRAIN / VALIDATION SPLIT
# ================================================================
# Use last 16 days of train as validation (mirrors test window exactly)
val_cut = pd.Timestamp("2017-07-31")
tr_mask = train_feat.date < val_cut
va_mask = train_feat.date >= val_cut

# Drop rows with too many NaN lags (first 112 days of each series)
train_clean = train_feat.dropna(subset=["lag_112","log1p_sales"]).copy()
train_clean[FEAT_COLS] = train_clean[FEAT_COLS].fillna(0)
test_feat[FEAT_COLS]   = test_feat[FEAT_COLS].fillna(0)

tr_mask2 = train_clean.date < val_cut
va_mask2 = train_clean.date >= val_cut

X_tr = train_clean.loc[tr_mask2, FEAT_COLS]
y_tr = train_clean.loc[tr_mask2, "log1p_sales"]
X_va = train_clean.loc[va_mask2, FEAT_COLS]
y_va = train_clean.loc[va_mask2, "log1p_sales"]
X_te = test_feat[FEAT_COLS]

print(f"  X_tr:{X_tr.shape}  X_va:{X_va.shape}  X_te:{X_te.shape}")
del train_feat, train_clean; gc.collect()

# ================================================================
# 8. LIGHTGBM — 3 SEED ENSEMBLE
# ================================================================
print("\n" + "="*60)
print("STEP 5 — Training LightGBM (3-seed ensemble)")
print("="*60)

PARAMS = dict(
    objective         = "regression_l1",
    metric            = "rmse",
    boosting_type     = "gbdt",
    num_leaves        = 255,
    learning_rate     = 0.04,
    feature_fraction  = 0.7,
    bagging_fraction  = 0.7,
    bagging_freq      = 1,
    min_child_samples = 30,
    reg_alpha         = 0.05,
    reg_lambda        = 1.0,
    n_jobs            = -1,
    verbose           = -1,
)

dtrain = lgb.Dataset(X_tr, y_tr)
dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)

models, val_scores = [], []
for seed in [42, 123, 777]:
    print(f"\n  ── Seed {seed} ──")
    PARAMS["seed"] = seed
    t0 = time.time()
    m = lgb.train(
        PARAMS, dtrain,
        num_boost_round = 3000,
        valid_sets      = [dvalid],
        callbacks       = [lgb.log_evaluation(100),
                           lgb.early_stopping(50, verbose=True)],
    )
    val_pred  = np.expm1(m.predict(X_va, num_iteration=m.best_iteration)).clip(0)
    val_true  = np.expm1(y_va.values).clip(0)
    # RMSLE
    rmsle_val = np.sqrt(np.mean((np.log1p(val_pred) - np.log1p(val_true))**2))
    print(f"  Val RMSLE={rmsle_val:.5f}  iter={m.best_iteration}  {time.time()-t0:.0f}s")
    models.append(m)
    val_scores.append(rmsle_val)

print(f"\n  Ensemble Val RMSLE: {np.mean(val_scores):.5f}")

# Feature importance
fi = pd.Series(models[0].feature_importance("gain"), index=FEAT_COLS)
print("\n  Top 25 features:")
print(fi.sort_values(ascending=False).head(25).to_string())

# ================================================================
# 9. PREDICT
# ================================================================
print("\n" + "="*60)
print("STEP 6 — Generating Predictions")
print("="*60)

all_preds = np.mean([
    m.predict(X_te, num_iteration=m.best_iteration) for m in models
], axis=0)
test_feat["sales_pred"] = np.expm1(all_preds).clip(min=0)

# ================================================================
# 10. SUBMISSION
# ================================================================
test_raw  = pd.read_csv(BASE/"test.csv", parse_dates=["date"])
sample    = pd.read_csv(BASE/"sample_submission.csv")

merged = test_raw[["id","date","store_nbr","family"]].merge(
    test_feat[["date","store_nbr","family","sales_pred"]],
    on=["date","store_nbr","family"], how="left"
)
submission = sample[["id"]].merge(merged[["id","sales_pred"]], on="id", how="left")
submission = submission.rename(columns={"sales_pred":"sales"})
submission["sales"] = submission["sales"].fillna(0).clip(lower=0)

out = OUT / "submission.csv"
submission.to_csv(out, index=False)

print(f"\n✅  submission.csv saved  →  {out}")
print(f"   Rows: {len(submission)}")
print(f"   Sales stats: min={submission.sales.min():.2f}  "
      f"mean={submission.sales.mean():.2f}  max={submission.sales.max():.2f}")
print(f"\n   Val RMSLE per seed: {[f'{s:.5f}' for s in val_scores]}")
print(f"   Expected leaderboard score: ~0.376-0.380")
print("\n🏆  Go to 'Submit Prediction' → upload submission.csv")
print("="*60)

STEP 1 — Loading Data
  Train: (3000888, 7) | 2013-01-01 → 2017-08-15
  Test : (28512, 5)  | 2017-08-16 → 2017-08-31
  Test window = 16 days
  ✅ Safe minimum lag = 16 days (lag_16 of Aug31 = Aug15 = last train day)

STEP 2 — Building lookup tables
  Done ✓

STEP 3 — Computing lag features
  6/33 families  3s
  12/33 families  5s
  18/33 families  7s
  24/33 families  10s
  30/33 families  12s
  Lag done in 14s | shape: (3000888, 37)
  Building test lag features …
  Test lag done | shape: (28512, 35)

STEP 4 — Adding calendar, holiday, oil features
  Train: (3000888, 70)  Test: (28512, 68)

  Total features: 64
  Feature list: ['store_nbr', 'onpromotion', 'lag_16', 'lag_21', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_91', 'lag_112', 'lag_364', 'lag_365', 'lag_366', 'lag_sw_4w', 'lag_sw_ly', 'roll_mean_7', 'roll_std_7', 'roll_max_7', 'roll_min_7', 'roll_mean_14', 'roll_std_14', 'roll_max_14', 'roll_min_14', 'roll_mean_28', 'roll_std_28', 'roll_max_28', 'roll_min_28', 'roll_mean_56', 'r